# Image Fundamentals

> 📘 **Python Mastery** · Module 15 — Computer Vision · Lesson 1/5

Every photo, screenshot and camera frame is just a grid of numbers — and once you see images that way,
everything in computer vision becomes ordinary NumPy. This lesson builds that mental model and fixes the
three bugs every beginner hits: sideways coordinates, broken colors, and overflowing pixels.

## 🎯 Learning Objectives

- **Explain** how grayscale (`H×W`) and color (`H×W×3`) images are stored as NumPy arrays
- **Navigate** OpenCV's coordinate system confidently: rows (`y`) go *down*, columns (`x`) go *right*
- **Create** your own images programmatically — gradients, shapes and text — instead of downloading them
- **Display** images correctly in matplotlib, including the famous BGR vs RGB trap
- **Crop, resize, flip and rotate** images, and choose the right interpolation flag
- **Convert** between BGR, grayscale (luma weights) and HSV, and isolate objects by color
- **Save** and reload images with `cv2.imwrite` / `cv2.imread` and verify nothing was lost

## 1. An Image Is Just a NumPy Array

A **grayscale** image is a 2-D matrix of shape `(height, width)`. Each cell is one **pixel**
(picture element) holding a brightness from `0` (black) to `255` (white).

A **color** image stacks three of those grids: shape `(height, width, 3)` — one channel each for
blue, green and red. Think of a mosaic made of tiny tiles: grayscale has one number per tile,
color has three.

**Syntax:**
```python
import numpy as np
gray = np.zeros((240, 320), dtype=np.uint8)       # H x W   -> grayscale
rgb  = np.zeros((240, 320, 3), dtype=np.uint8)    # H x W x 3 -> color
print(gray.shape, gray.dtype)   # (240, 320) uint8
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# A tiny 5x4 grayscale "image" so you can SEE the numbers become a picture
tiny = np.array([[  0,  60, 120, 180],
                 [ 30,  90, 150, 210],
                 [255, 200, 140,  80],
                 [220, 160, 100,  40],
                 [190, 130,  70,  10]], dtype=np.uint8)

print("shape :", tiny.shape)     # (rows=y, cols=x)
print("dtype :", tiny.dtype)     # uint8 = unsigned 8-bit integer
print("values:", tiny.min(), "..", tiny.max())
print(tiny)                      # the picture IS this table of numbers

plt.figure(figsize=(3, 3))
plt.imshow(tiny, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
plt.title("Same numbers, rendered as pixels")
plt.axis("off")
plt.show()

> 🔍 **Under the Hood:** `uint8` means *unsigned integer, 8 bits* — exactly **one byte per pixel
> per channel**. A 1920×1080 color frame is therefore `1920 × 1080 × 3 = 6,220,800 bytes ≈ 6 MB`.
> That's why video pipelines care so much about dtype: processing frames as `float64` silently makes
> them **8× bigger** in RAM. OpenCV functions expect `uint8` input and will raise errors (or behave
> oddly) if you hand them floats without converting back.

## 2. The Coordinate System: Rows Go *Down*

NumPy indexes arrays `[row, column]`, and OpenCV follows suit: `img[y, x]`.
Unlike school geometry, the origin `(0, 0)` is the **top-left** corner and `y` grows **downward**.
Mixing up `x` and `y` is the single most common CV bug — remember: **shape is `(height, width)`,
index is `[y, x]`**.

| index | x = 0 | x = 1 | x = 2 | x = 3 |
|-------|-------|-------|-------|-------|
| **y = 0** | 0 | 60 | 120 | 180 |
| **y = 1** | 30 | 90 | 150 | 210 |
| **y = 2** | 255 | 200 | 140 | 80 |

**Syntax:**
```python
pixel = img[y, x]              # one pixel (scalar for gray, 3-vector for color)
region = img[y1:y2, x1:x2]     # a rectangular patch (crop)
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

img = np.full((200, 300), 40, dtype=np.uint8)        # dark canvas: 200 rows, 300 cols

x, y = 220, 60                                       # x = 220th column, y = 60th ROW (from top!)
img[y, x] = 255                                      # paint ONE pixel white

plt.figure(figsize=(4, 3))
plt.imshow(img, cmap="gray")
plt.scatter([x], [y], s=260, facecolors="none", edgecolors="red", linewidths=2)  # highlight it
plt.title(f"Pixel at (x={x}, y={y}) -> img[{y}, {x}]")
plt.xlabel("x  (columns, left->right)")
plt.ylabel("y  (rows, TOP -> bottom)")
plt.show()

print("Value there:", img[y, x])                     # 255

## 3. `uint8` Overflow: Why 200 + 100 = 44

`uint8` can hold `0…255`. Add beyond 255 and NumPy **wraps around** (modulo 256): `200 + 100 = 300`,
and `300 mod 256 = 44`. Your "brighter" image suddenly goes dark. OpenCV's own arithmetic
functions **saturate** instead — they clamp at 255. When doing manual math, work in a wider dtype
(`int16` / `float32`), clip, then cast back.

**Syntax:**
```python
bright = cv2.add(img, 80)                          # saturates at 255 (safe)
tmp    = img.astype(np.int16) + 80                 # widen first...
bright = np.clip(tmp, 0, 255).astype(np.uint8)     # ...clip, then narrow
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

base = np.full((120, 200), 200, dtype=np.uint8)      # light-gray canvas

wrapped = base + 100                                  # uint8 math: WRAPS to 44!
safe    = cv2.add(base, 100)                          # OpenCV: SATURATES at 255
manual  = np.clip(base.astype(np.int16) + 100, 0, 255).astype(np.uint8)

print("numpy uint8 : 200 + 100 =", wrapped.ravel()[0])            # 44  (!!)
print("cv2.add     : 200 + 100 =", safe.ravel()[0])               # 255
print("manual clip : 200 + 100 =", manual.ravel()[0])             # 255

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, im, ttl in zip(axes, [wrapped, safe, manual],
                       ["WRONG: base + 100 (wraps)", "cv2.add (saturates)", "int16 -> clip -> uint8"]):
    ax.imshow(im, cmap="gray", vmin=0, vmax=255)
    ax.set_title(ttl, fontsize=9)
    ax.axis("off")
fig.suptitle("The 200 + 100 = 44 trap")
plt.show()

## 4. Drawing Your Own Images (No Downloads Needed)

You rarely need stock photos to practice. Start from a blank canvas with `np.zeros` / `np.full`,
then paint with OpenCV's drawing functions. **Careful:** OpenCV colors are written **`(B, G, R)`**,
not `(R, G, R)` — pure red is `(0, 0, 255)`. A `thickness` of `-1` fills the shape.

**Syntax:**
```python
canvas = np.zeros((H, W, 3), dtype=np.uint8)
cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thickness)
cv2.circle(canvas, (cx, cy), radius, color, thickness)
cv2.putText(canvas, "text", (x, y), font, scale, color, thickness)
```

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

Path("sample_data").mkdir(exist_ok=True)

W, H = 320, 240
row = np.linspace(35, 110, W).astype(np.uint8)         # one row getting brighter
sky = np.tile(row, (H, 1))                             # repeat down -> smooth gradient
scene = cv2.cvtColor(sky, cv2.COLOR_GRAY2BGR)          # promote to 3 channels (BGR!)

cv2.rectangle(scene, (0, 190), (W, H), (30, 90, 40), -1)            # grass (BGR dark green)
cv2.circle(scene, (262, 52), 26, (0, 215, 255), -1)                 # sun (yellow)
cv2.rectangle(scene, (60, 110), (150, 195), (60, 60, 120), -1)      # house wall (brownish)
pts = np.array([[50, 112], [105, 62], [160, 112]], np.int32)        # roof triangle
cv2.fillPoly(scene, [pts], (40, 40, 170))
cv2.rectangle(scene, (92, 145), (118, 195), (40, 40, 200), -1)      # door (BGR red-ish)
cv2.rectangle(scene, (66, 122), (84, 138), (230, 230, 230), -1)     # window
cv2.putText(scene, "My first scene", (8, 24),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)

cv2.imwrite("sample_data/street_scene.png", scene)     # saved for later lessons
print("Saved sample_data/street_scene.png:", scene.shape)

plt.figure(figsize=(5, 4))
plt.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))     # BGR -> RGB before showing!
plt.title("A scene built from pure NumPy + OpenCV")
plt.axis("off")
plt.show()

## 5. Displaying Images Correctly (the #1 Beginner Bug)

OpenCV stores color images as **BGR**; matplotlib assumes **RGB**. Show a BGR array directly and
every color flips to its opposite: red objects turn blue, blue turns orange-yellow.
The fix is one line — `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` — applied *only for display*
(keep working in BGR for OpenCV functions). Grayscale needs `cmap="gray"`, otherwise matplotlib
paints it with the default *viridis* rainbow colormap.

**Syntax:**
```python
plt.imshow(gray_img, cmap="gray")                              # grayscale
plt.imshow(cv2.cvtColor(bgr_img, cv2.COLOR_BGR2RGB))           # color, correct order
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

canvas = np.zeros((220, 320, 3), dtype=np.uint8)
cv2.rectangle(canvas, (30, 40), (130, 180), (0, 0, 255), -1)    # RED ball  (BGR: B=0,G=0,R=255)
cv2.rectangle(canvas, (180, 40), (290, 180), (255, 0, 0), -1)   # BLUE box  (BGR: B=255,...)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].imshow(canvas)                                          # WRONG: raw BGR shown as RGB
axes[0].set_title("WRONG: imshow(bgr) -- colors swapped")

axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))         # RIGHT
axes[1].set_title("RIGHT: convert BGR -> RGB first")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Left panel: the 'red' square LOOKS blue. Nothing changed in the data -")
print("only the interpretation of the channels did.")

## 6. Pixel Access, Channels & Cropping

Read or write any pixel with `img[y, x]`; grab a whole color channel with `img[:, :, c]`
(recall: in BGR, channel **0 = blue**, channel **2 = red**). A **crop** is plain NumPy slicing:
`img[y1:y2, x1:x2]` — rows first, and the stop index is exclusive.

**Syntax:**
```python
b, g, r = img[100, 50]                  # unpack one pixel's three channels
blue_ch = img[:, :, 0]                  # whole blue plane (H x W, 2-D)
face   = img[40:200, 120:360]           # rectangular crop
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

scene = cv2.imread("sample_data/street_scene.png")       # built in section 4
assert scene is not None, "Run section 4 first!"

print("Pixel in the sky  scene[20, 200] =", scene[20, 200])
print("Pixel on the door scene[170, 104] =", scene[170, 104])

door_crop = scene[130:200, 85:125]                        # [y1:y2, x1:x2]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB))
axes[0].add_patch(plt.Rectangle((85, 130), 40, 70, fill=False, color="lime", lw=2))
axes[0].set_title("Scene with crop region marked")
axes[1].imshow(cv2.cvtColor(door_crop, cv2.COLOR_BGR2RGB))
axes[1].set_title("scene[130:200, 85:125]")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Channel planes: which channel "sees" the red door best?
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for i, name in enumerate(["Blue ch", "Green ch", "Red ch"]):
    axes[i].imshow(scene[:, :, i], cmap="gray")
    axes[i].set_title(name)
    axes[i].axis("off")
plt.suptitle("Channel slices of a BGR image (0=B, 1=G, 2=R)")
plt.show()

> 🔍 **Under the Hood:** Slicing returns a **view**, not a copy — `door = img[0:50, 0:50]`
> shares memory with `img`, so `door[:] = 0` punches a black hole in the original. That's usually what
> you want (zero copying for big frames), but if you plan to modify a crop independently, call
> `.copy()`. Also note rows are stored contiguously (C order), so `img[i]` (a whole row) is very fast
> while stepping down columns jumps through memory — one reason OpenCV algorithms iterate rows.

## 7. Resizing & Choosing an Interpolation

`cv2.resize` takes the target size as **`(width, height)`** — width first, the opposite order from
`shape`! The `interpolation` flag decides *how* new pixel values are invented:

| Flag | Strategy | Best for |
|------|----------|----------|
| `INTER_NEAREST` | copy closest pixel | speed, masks/labels (keeps integers) |
| `INTER_LINEAR` | weighted average of 2×2 neighbours | default, most enlargements |
| `INTER_CUBIC` | weighted 4×4 spline | enlargements, sharper than linear |
| `INTER_AREA` | area-weighted average | **shrinking** (anti-aliasing) |
| `INTER_LANCZOS4` | 8×8 window | highest-quality enlargement, slower |

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

scene = cv2.imread("sample_data/street_scene.png")

tiny = cv2.resize(scene, (56, 42), interpolation=cv2.INTER_AREA)   # crush to 56x42
flags = [("INTER_NEAREST", cv2.INTER_NEAREST), ("INTER_LINEAR", cv2.INTER_LINEAR),
         ("INTER_CUBIC", cv2.INTER_CUBIC), ("INTER_LANCZOS4", cv2.INTER_LANCZOS4)]

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
axes[0, 0].imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB)); axes[0, 0].set_title("Original 320x240")
for ax, (name, f) in zip(axes.ravel()[1:], flags):
    back = cv2.resize(tiny, (320, 240), interpolation=f)
    ax.imshow(cv2.cvtColor(back, cv2.COLOR_BGR2RGB))
    ax.set_title(f"56x42 -> 320x240\n{name}")
for ax in axes.ravel():
    ax.axis("off")
plt.suptitle("Same tiny image, four ways to blow it back up")
plt.show()

# Shrinking aliasing demo: a checkerboard collapsed by NEAREST vs AREA
board = np.kron([[1, 0] * 8, [0, 1] * 8] * 8, np.ones((12, 12))).astype(np.float32) * 255
near = cv2.resize(board, (32, 32), interpolation=cv2.INTER_NEAREST)
area = cv2.resize(board, (32, 32), interpolation=cv2.INTER_AREA)

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(near, cmap="gray");  axes[0].set_title("Shrink w/ NEAREST: moire artifacts")
axes[1].imshow(area, cmap="gray");  axes[1].set_title("Shrink w/ AREA: clean average")
for ax in axes:
    ax.axis("off")
plt.show()

## 8. Flipping & Rotating

Flips mirror the image (`flipCode`: `1` = horizontal, `0` = vertical, `-1` = both) — the cheapest
data augmentation there is. `cv2.rotate` handles exact 90°/180° turns; for arbitrary angles build a
rotation matrix with `cv2.getRotationMatrix2D(center, angle, scale)` and apply it with
`cv2.warpAffine`.

**Syntax:**
```python
flipped = cv2.flip(img, 1)                                   # mirror left-right
turned  = cv2.rotate(img, cv2.ROTATE_90_CLOCKWISE)
M = cv2.getRotationMatrix2D((cx, cy), angle_deg, 1.0)        # 2x3 transform matrix
rotated = cv2.warpAffine(img, M, (w, h), borderValue=color)
```

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

scene = cv2.imread("sample_data/street_scene.png")
H, W = scene.shape[:2]

mirror = cv2.flip(scene, 1)                                   # horizontal flip
quarter = cv2.rotate(scene, cv2.ROTATE_90_CLOCKWISE)          # exact 90 degrees
M = cv2.getRotationMatrix2D((W / 2, H / 2), 30, 1.0)          # 30 deg around center
tilted = cv2.warpAffine(scene, M, (W, H),
                        flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))

views = [(scene, "Original"), (mirror, "cv2.flip(scene, 1)"),
         (quarter, "ROTATE_90_CLOCKWISE"), (tilted, "warpAffine, 30 deg")]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, (im, ttl) in zip(axes, views):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(ttl, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Original shape:", scene.shape, "-> rotated 90deg shape:", quarter.shape)

## 9. Grayscale Conversion: The Luma Weights

Throwing channels away isn't enough — you must **mix** them. OpenCV uses the ITU-R BT.601 luma
weights, chosen to match human eye sensitivity (we perceive green as brightest, blue as darkest):

$$Y = 0.299\,R + 0.587\,G + 0.114\,B$$

`cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` applies exactly that (note the channel order when you
compute it yourself!). Many algorithms — Canny, thresholding, Haar cascades — require grayscale
input anyway, so this conversion opens almost every lesson that follows.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

scene = cv2.imread("sample_data/street_scene.png")

# Do it manually to prove the formula (mind the B-G-R order!)
B = scene[:, :, 0].astype(np.float32)
G = scene[:, :, 1].astype(np.float32)
R = scene[:, :, 2].astype(np.float32)
manual = (0.114 * B + 0.587 * G + 0.299 * R)

gray = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
diff = np.abs(np.round(manual) - gray.astype(np.float32)).max()
print("Max difference manual vs cv2.cvtColor:", diff, "-> same result (rounding only)")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB)); axes[0].set_title("Color")
axes[1].imshow(gray, cmap="gray", vmin=0, vmax=255);    axes[1].set_title("Grayscale (luma)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("shapes:", scene.shape, "->", gray.shape)

## 10. HSV: Color That Survives Lighting

RGB mixes *"which color"* with *"how bright"* in every channel, so simple red-vs-blue checks break
whenever shadows move. **HSV** separates them: **H**ue (the color itself, 0–179 in OpenCV),
**S**aturation (colorfulness), **V**alue (brightness). To isolate a color: convert → mask with
`cv2.inRange` → combine with `cv2.bitwise_and`.

| Color | Approx. OpenCV hue range (0–179) |
|-------|-------------------------------|
| Red   | 0–10 **and** 170–179 (it wraps!) |
| Green | 35–85 |
| Blue  | 95–135 |

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# A little "fruit basket": blue, green and red balls on a dark tray
basket = np.full((220, 340, 3), 35, dtype=np.uint8)
for cx, cy, col in [(70, 110, (255, 0, 0)), (170, 130, (0, 180, 0)), (270, 105, (0, 0, 255))]:
    cv2.circle(basket, (cx, cy), 46, col, -1)
    cv2.ellipse(basket, (cx, cy - 46), (18, 8), 0, 0, 360, (90, 90, 90), -1)  # little stem shadow
cv2.putText(basket, "pick the BLUE balls", (55, 205), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

hsv = cv2.cvtColor(basket, cv2.COLOR_BGR2HSV)                       # H: 0-179, S,V: 0-255
mask = cv2.inRange(hsv, (95, 80, 80), (135, 255, 255))              # blue hue band
only_blue = cv2.bitwise_and(basket, basket, mask=mask)
print("Mask covers %.1f%% of pixels" % (100 * (mask > 0).mean()))

fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
panels = [(basket, "Basket (shown as RGB)"), (mask, "cv2.inRange blue mask"), (only_blue, "bitwise_and: isolated")]
for ax, (im, ttl) in zip(axes, panels):
    ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
    if im.ndim == 3:
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    ax.set_title(ttl, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 11. Saving & Reading Image Files

`cv2.imwrite(path, img)` infers the format from the extension; `cv2.imread(path)` returns a BGR
array — or **`None` if anything went wrong** (missing file, bad permission, unsupported format). It
does *not* raise an exception, so always check. PNG is **lossless**: reloading gives back identical
bytes. JPEG trades fidelity for size — perfect for photos, bad for pixel-exact tests.

In [ ]:
import numpy as np
import cv2
from pathlib import Path
import matplotlib.pyplot as plt

Path("sample_data").mkdir(exist_ok=True)

logo = np.zeros((160, 200, 3), dtype=np.uint8)
cv2.circle(logo, (100, 70), 45, (0, 140, 255), -1)
cv2.putText(logo, "PyMastery", (28, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

ok_png = cv2.imwrite("sample_data/logo.png", logo)                    # lossless
ok_jpg = cv2.imwrite("sample_data/logo.jpg", logo, [cv2.IMWRITE_JPEG_QUALITY, 40])  # lossy
png_back = cv2.imread("sample_data/logo.png")
jpg_back = cv2.imread("sample_data/logo.jpg")

print("imwrite returned:", ok_png, ok_jpg)
if png_back is None:                                                  # ALWAYS check!
    raise RuntimeError("Reload failed - check the path")

print("PNG identical to original? ", np.array_equal(logo, png_back))
print("JPG identical to original? ", np.array_equal(logo, jpg_back),
      "| worst pixel drift:", int(np.abs(logo.astype(int) - jpg_back).max()))

fig, axes = plt.subplots(1, 3, figsize=(11, 3))
axes[0].imshow(logo[..., ::-1]);                axes[0].set_title("Original")
axes[1].imshow(png_back[..., ::-1]);            axes[1].set_title("PNG reload (lossless)")
drift = cv2.applyColorMap(cv2.convertScaleAbs(cv2.absdiff(logo, jpg_back), alpha=6), cv2.COLORMAP_INFERNO)
axes[2].imshow(drift[..., ::-1]);               axes[2].set_title("JPEG error map (quality 40)")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---------|---------|-----|
| `plt.imshow(gray)` without `cmap="gray"` | Matplotlib paints it with the viridis rainbow | `plt.imshow(gray, cmap="gray")` |
| Showing BGR data directly | Colors invert: red looks blue | `plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))` |
| `img[x, y]` | Shape is `(H, W)`, so this swaps axes — IndexError or silent wrong pixel | Index as `img[y, x]`; `resize` wants `(width, height)` |
| `img + 100` on `uint8` | Wraparound: `200 + 100 = 44` | `cv2.add(img, 100)` or widen → clip → cast back |
| Ignoring `cv2.imread` returning `None` | Crash much later with a cryptic error | `assert img is not None, "path?"` right after loading |
| Editing a slice expecting independence | Slices are views: the original changes too | `crop = img[y1:y2, x1:x2].copy()` |

## 💡 Best Practices & Pro Tips

- **Convert once, at the border.** Load → sanity-check `is not None` → make an RGB copy for
  display, keep the BGR original for OpenCV work. Never convert back and forth mid-algorithm.
- **Keep an untouched master copy** (`original = img.copy()`) — destructive experiments are then free.
- **Do math in `float32`, store in `uint8`.** Widen → operate → clip → cast is the universal idiom.
- **Generate synthetic fixtures** into `sample_data/` (as we did) — reproducible, license-free, and
  they make failures easy to debug because you know the ground truth.
- 🤖 **AI-engineering relevance:** every dataset pipeline (PIL/torchvision, tf.data, albumentations)
  ends with exactly these operations — decode, channel reorder, resize with the right interpolation,
  normalize dtype. A wrong interpolation flag or a forgotten BGR→RGB swap won't crash anything;
  it will quietly cost your model accuracy points. Understanding the array is debugging power.

## 📌 Summary

| Method | What it does | Example |
|--------|--------------|---------|
| `np.zeros((H, W), np.uint8)` | Blank grayscale canvas | `img = np.zeros((240, 320), np.uint8)` |
| `img[y, x]` / `img[y1:y2, x1:x2]` | Pixel / region access | `patch = img[40:120, 60:200]` |
| `cv2.cvtColor(img, cv2.COLOR_BGR2RGB)` | Reorder channels for matplotlib | `plt.imshow(...)` |
| `cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` | Luma-weighted grayscale | `gray = cvtColor(img, COLOR_BGR2GRAY)` |
| `cv2.resize(img, (w, h), interp)` | Scale (width first!) | `cv2.resize(img, (320, 240), interpolation=cv2.INTER_AREA)` |
| `cv2.flip` / `cv2.rotate` / `warpAffine` | Flip, 90° turns, arbitrary rotation | `cv2.flip(img, 1)` |
| `cv2.inRange(hsv, lo, hi)` | Binary color mask in HSV | `cv2.inRange(hsv, (95,80,80), (135,255,255))` |
| `cv2.imwrite` / `cv2.imread` | Save / load (PNG = lossless) | `cv2.imwrite("out.png", img)` |

- An image is a `(H, W)` or `(H, W, 3)` `uint8` array — rows go down, index is `[y, x]`.
- Display grayscale with `cmap="gray"` and always convert BGR→RGB for color.
- `uint8` overflows by wrapping; `cv2.add` and friends saturate instead.
- Choose interpolation deliberately: `INTER_AREA` to shrink, `LINEAR`/`CUBIC` to grow.
- HSV separates hue from brightness — the classic trick for robust color isolation.

## 🔗 Next Lesson

Continue to **[02_Image_Processing_OpenCV](../02_Image_Processing_OpenCV/notes.ipynb)** — brightness,
contrast, thresholding, blurring, sharpening and morphology: the classic toolbox for making messy
images clean.